# Détection de profils atypiques sur Twitter — Récap

## 1. Contexte et objectif

Comparaison de **deux approches de classification** pour détecter des profils **atypiques** sur Twitter :

| Approche | Méthodes comparées | Retenue | Notebook |
|----------|-------------------|---------|----------|
| Non supervisée | K-Means vs Isolation Forest | Isolation Forest | `Groupe3_profils_atypiques_non_Sup.ipynb` |
| Supervisée | SVM vs XGBoost | XGBoost | `Groupe3_profils_atypiques_Sup.ipynb` |

**Jeu de données :** `users_labeled_manual.csv` (643 124 profils, ~17 % atypiques)

Source : ~1,16 M tweets agrégés par utilisateur (MongoDB).

## 2. Méthodologie commune

**Pipeline :** features -> `StandardScaler` -> **ACP** -> modèles

| Contexte | Features | Composantes ACP |
|----------|----------|-----------------|
| Non supervisé (16 feat.) | Seuil 75 % variance | **7 (~79 %)** |
| Supervisé (8 feat.) | Saturation 100 % | **5** |

> **k=7** K-Means = nombre de **clusters** (coude inertie), indépendant de l'ACP.

| Étape | Non supervisé | Supervisé |
|-------|---------------|-----------|
| Features | 16 MongoDB | 8 hors règles Excel |
| Normalisation | StandardScaler | StandardScaler (fit train) |
| ACP | 7 composantes | 5 composantes |
| Modèles | K-Means, Isolation Forest | SVM, XGBoost |

**8 features supervisées (hors règles) :** `followers_count`, `friends_count`, `avg_tweet_length`, `avg_hashtags`, `avg_favorites`, `avg_retweet_count`, `verified`, `default_profile_image`.

## 3. Approche non supervisée — K-Means vs Isolation Forest

### 3.1 K-Means (MiniBatchKMeans, k = 7 clusters)

- Clustering sur l'espace ACP (**7 composantes**, ~79 % variance)
- Profils atypiques = clusters minoritaires (< 1 %)
- Résultat : ~3 498 profils (0,54 %)

### 3.2 Isolation Forest

- `contamination = 'auto'` (seuil déterminé par les scores d'anomalie)
- Résultat : ~42 987 profils (6,68 %)

### 3.3 Méthode retenue : Isolation Forest

Conçue pour la détection d'anomalies, plus sensible que K-Means sur 643 000 profils.

## 4. Approche supervisée — SVM vs XGBoost

Split train/test 80/20 stratifié. Pipeline : StandardScaler -> **ACP (5 comp.)** -> modèle.

### Limite méthodologique

Les labels suivent 4 règles Excel. Avec **16 features + ACP 7 comp.**, F1 ~ 0,970 (circularité).
Évaluation retenue : **8 features hors règles + ACP 5 comp.**

### 4.3 Comparaison (ACP 5 comp. + 8 features hors règles)

| Métrique | SVM | XGBoost |
|----------|-----|---------|
| Accuracy | 55,9 % | **66,7 %** |
| F1 (Atypique) | 0,361 | **0,443** |
| Recall | 0,737 | **0,783** |
| Precision | 0,239 | **0,309** |
| ROC-AUC | 0,670 | **0,794** |

### Modèle retenu : XGBoost

Meilleur F1 et ROC-AUC hors règles de labellisation.

## 5. Comparaison des deux approches retenues

| | Isolation Forest | XGBoost |
|--|------------------|---------|
| Labels requis | Non | Oui (Excel) |
| Features ML | 16 | 8 (hors règles) |
| ACP | 7 composantes | 5 composantes |
| Détection / F1 vs labels | ~42 987 (6,68 %) | F1 = 0,443 |
| Forces | Exploration sans annotation | Signal indirect avec labels |
| Faiblesses | Seuil auto (non contrôlé) | Label corrélé aux règles Excel |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Synthèse visuelle des méthodes retenues
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Non supervisé : répartition des détections
categories_ns = ["K-Means\nseulement", "Consensus\n(K-Means + Iso)", "Iso Forest\nseulement"]
values_ns = [0, 3498, 39489]
colors_ns = ["#2196F3", "#9C27B0", "#F44336"]
axes[0].bar(categories_ns, values_ns, color=colors_ns, edgecolor="black")
axes[0].set_title("Non supervisé — Profils atypiques détectés\n(Isolation Forest retenu)", fontweight="bold")
axes[0].set_ylabel("Nombre de profils")
axes[0].grid(axis="y", alpha=0.4)
for i, v in enumerate(values_ns):
    if v > 0:
        axes[0].text(i, v + 500, f"{v:,}", ha="center", fontweight="bold")

# Supervisé : F1-score
models = ["SVM", "XGBoost"]
f1_scores = [0.361, 0.443]
bars = axes[1].bar(models, f1_scores, color=["#2196F3", "#4CAF50"], edgecolor="black", width=0.5)
axes[1].set_title("Supervisé — F1 (ACP 5 comp., 8 feat.)\n(XGBoost retenu)", fontweight="bold")
axes[1].set_ylabel("F1-score")
axes[1].set_ylim(0, 1.05)
axes[1].grid(axis="y", alpha=0.4)
for bar, val in zip(bars, f1_scores):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f"{val:.3f}", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# Tableau récapitulatif final
recap = pd.DataFrame({
    "Approche": ["Non supervisée", "Supervisée"],
    "Méthodes comparées": ["K-Means vs Isolation Forest", "SVM vs XGBoost"],
    "Méthode retenue": ["Isolation Forest", "XGBoost"],
    "ACP / Détection": ["7 comp. — ~42 987 profils (6,68 %)", "5 comp. — F1 = 0,443"],
    "Notebook": [
        "Groupe3_profils_atypiques_non_Sup.ipynb",
        "Groupe3_profils_atypiques_Sup.ipynb"
    ]
})

print("=" * 70)
print(" SYNTHÈSE FINALE — MÉTHODES RETENUES")
print("=" * 70)
display(recap)

## 6. Conclusion

Quatre méthodes comparées en deux approches complémentaires :

1. **Isolation Forest** (non supervisée) — ~6,7 % de profils déviants sans labels (contamination auto). ACP **7 composantes** (16 features).

2. **XGBoost** (supervisée) — F1 = 0,443, ROC-AUC = 0,794. ACP **5 composantes** (8 features hors règles Excel).

**Recommandation :** Isolation Forest en exploration ; XGBoost une fois les labels définis, sans réutiliser les variables des règles Excel en entrée.

---

*Projet IF29 — Groupe 3*